# Spotify financial data cleaning
This notebook reproduces and reviews the Module 6 cleaning pipeline. The business question is how Spotify's revenue growth, margins, profitability, cash generation, and liquidity changed over time.

In [ ]:
from pathlib import Path
import csv, json, subprocess, sys

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'clean_spotify_financials.py').exists():
    raise RuntimeError('Start Jupyter in the Stock_Movement_Context_Dashboard directory')
subprocess.run([sys.executable, 'scripts/clean_spotify_financials.py'], check=True)

## Audit evidence
The audit reconciles source and final counts and records missingness and conversion failures.

In [ ]:
audit = json.loads((ROOT / 'reports' / 'cleaning_audit.json').read_text())
audit

In [ ]:
assert audit['selected_source_rows'] == 152
assert audit['repeated_business_key_rows_removed'] == 91
assert audit['final_rows'] == audit['unique_record_ids'] == 61
assert audit['date_conversion_failures'] == 0
assert audit['groups_with_conflicting_values_across_filings'] == 0
print('Count reconciliation: 152 - 91 = 61; all checks passed.')

## Clean annual analysis table
Values are euros. Margins and growth are percentages; 2016 YoY is blank because the selected annual revenue series starts in 2016.

In [ ]:
with (ROOT / 'data' / 'processed' / 'spotify_year_summary.csv').open(newline='', encoding='utf-8') as f:
    annual = list(csv.DictReader(f))
annual

## Five record checks
These expectations were declared in the cleaning script before the output comparison.

In [ ]:
with (ROOT / 'reports' / 'five_record_checks.csv').open(newline='', encoding='utf-8') as f:
    checks = list(csv.DictReader(f))
assert len(checks) == 5 and all(row['matches'] == 'True' for row in checks)
checks

## Reproducibility tests

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)